In [112]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [113]:
import sys 
from pathlib import Path


project_root = None
for p in Path.cwd().resolve().parents:
    if (p / "utils").exists() and (p / "data").exists():
        project_root = p
        break

if project_root is None:
    raise RuntimeError("Raíz no encontrada.")

sys.path.insert(0, str(project_root))

In [114]:
import logging
import numpy as np
import pandas as pd
from utils.paths import find_project_root

logging.basicConfig(
	level=logging.INFO,
	format="%(asctime)s | %(levelname)s | %(message)s"
)

project_root = None
for p in Path.cwd().resolve().parents:
    if (p / "utils").exists() and (p / "data").exists():
        project_root = p
        break

if project_root is None:
    logging.error("No se encontró la raíz del proyecto")
    raise RuntimeError("Raíz no encontrada.")

logging.info(f"Raiz encontrada en {project_root}")
sys.path.insert(0, str(project_root))

2025-12-02 12:24:39,603 | INFO | Raiz encontrada en C:\Users\gianlu\Market-Scraper\Market-Scraper


### Extraemos y vemos la información pura del csv almacenado

In [115]:
root = find_project_root()
csv_path = root / "data" / "raw" / "job_market.csv"


df = pd.read_csv(csv_path)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18039 entries, 0 to 18038
Data columns (total 35 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   job_url                18039 non-null  object 
 1   company                17817 non-null  object 
 2   company_addresses      5768 non-null   object 
 3   company_description    4315 non-null   object 
 4   company_industry       1874 non-null   object 
 5   company_logo           12590 non-null  object 
 6   company_num_employees  6056 non-null   object 
 7   company_rating         0 non-null      float64
 8   company_revenue        4413 non-null   object 
 9   company_reviews_count  0 non-null      float64
 10  company_url            16891 non-null  object 
 11  company_url_direct     6392 non-null   object 
 12  created_at             18039 non-null  object 
 13  currency               6928 non-null   object 
 14  date_posted            18027 non-null  object 
 15  de

C:\Users\gianlu\AppData\Local\Temp\ipykernel_16312\2831658031.py:5: DtypeWarning: Columns (21,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


In [116]:
raw_backup_path = root / "data" / "raw" / "job_market_raw_backup.csv"
df.to_csv(raw_backup_path, index=False)

### Tratamiento de datos almacenados dentro del csv 

In [117]:
def drop_columns(df):
    cols_to_drop=['company_rating','company_reviews_count', 'experience_range', 'skills', 'vacancy_count', 'work_from_home_type']
    existing = [cols for cols in cols_to_drop if cols in df.columns]
    df = df.drop(columns=existing)
    return df
    

In [118]:
def dedupe_jobs(df):
    key = 'id' if 'id' in df.columns else 'job_url'
    if 'created' in df.columns:
        df = df.sort_values('created', ascending=False)
    df = df.drop_duplicates(subset=[key], keep='first')
    return df

In [119]:
def normalize_string_column(df, col):
    if col not in df.columns:
        return df
    
    df[col] = df[col].replace([float("inf"), float("-inf")], None)
    
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .replace({"nan": None, "None": None, "": None, "inf": None, "-inf": None})
    )
    df[col] = df[col].where(df[col].notna(), None)
    return df

In [120]:
def rename_columns(df):
    rename_mapping = {
                        'job_url':'url',
                        'company_addresses':'address',
                        'company_industry':'industry',
                        'company_logo':'logo',
                        'company_num_employees':'num_employees',
                        'company_rating':'rating',
                        'company_revenue':'revenue',
                        'company_reviews_count':'reviews_count',
                        'job_url_direct':'url_direct',
                        'created_at':'created',
                        'is_remote':'remote',
                        'job_function': 'role',
                        'job_level':'seniority',
                        'listing_type':'listing',
                        'title':'job_title'
                     }
    rename_map = {k: v for k, v in rename_mapping.items() if k in df.columns}
    df = df.rename(columns=rename_map)
    return df


In [121]:
def change_values(df):
  date_cols = ["created", "date_posted"]
  for col in date_cols:
    if col in df.columns:
      df[col] = pd.to_datetime(
        df[col],
        errors="coerce",
        utc=True
      )
    df = df.replace([float("inf"), float("-inf")], None)
    df = df.where(df.notna(), None)
  return df

In [122]:
def convert_timestamps(df):
    for col in df.columns:
        if pd.api.types.is_datetime64tz_dtype(df[col]):
            df[col] = df[col].dt.strftime("%Y-%m-%d %H:%M:%S")
    return df

In [123]:
def add_missing_flags(df, columns):
    for col in columns:
        if col in df.columns:
            is_all_na = df[col].isna().all()
            is_any_na = df[col].isna().any()
            if is_any_na and not is_all_na:
                df[f"{col}_missing"] = df[col].isna().astype(int)
    return df

## Muestra el df limpio

In [124]:
df_proccesed = drop_columns(df)
df_proccesed = dedupe_jobs(df_proccesed)
columns_to_normalize = ["company", "address", "company_description", "industry", 
                        "logo", "num_employees", "revenue", "company_url", 
                        "company_url_direct", "currency", "date_posted", "description", 
                        "emails", "interval", "role", "seniority", "job_type", "url_direct", 
                        "listing", "location", "min_amount", "max_amount"]
for col in columns_to_normalize:
	df_proccesed = normalize_string_column(df_proccesed, col)
df_proccesed = rename_columns(df_proccesed)
df_proccesed = change_values(df_proccesed)
df_proccesed = convert_timestamps(df_proccesed)
df_proccesed = add_missing_flags(df_proccesed, ["company", "company_url", "location", "max_amount", "min_amount", "listing"])  

df_proccesed.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15920 entries, 0 to 18038
Data columns (total 35 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   url                  15920 non-null  object
 1   company              15698 non-null  object
 2   address              5766 non-null   object
 3   company_description  4308 non-null   object
 4   industry             1874 non-null   object
 5   logo                 10797 non-null  object
 6   num_employees        6049 non-null   object
 7   revenue              4411 non-null   object
 8   company_url          14964 non-null  object
 9   company_url_direct   6385 non-null   object
 10  created              15920 non-null  object
 11  currency             5314 non-null   object
 12  date_posted          12680 non-null  object
 13  description          10904 non-null  object
 14  emails               1890 non-null   object
 15  id                   15920 non-null  object
 16  interval 

C:\Users\gianlu\AppData\Local\Temp\ipykernel_16312\2024581391.py:3: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if pd.api.types.is_datetime64tz_dtype(df[col]):
C:\Users\gianlu\AppData\Local\Temp\ipykernel_16312\2024581391.py:3: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if pd.api.types.is_datetime64tz_dtype(df[col]):
C:\Users\gianlu\AppData\Local\Temp\ipykernel_16312\2024581391.py:3: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if pd.api.types.is_datetime64tz_dtype(df[col]):


In [125]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

root = find_project_root()


csv_path = root / "data" / "processed" / "job_market_processed.csv"
csv_path.parent.mkdir(parents=True, exist_ok=True)

logging.info(f"Ruta {root} encontrada.")
logging.info(f"CSV exportado hacia {csv_path}")

job_processed_path = root / "data" / "processed" / "job_market_processed.csv" 
df_proccesed.to_csv(job_processed_path, index=False, encoding="utf-8")

logging.info(f"CSV creado en {csv_path.relative_to(root)}")


2025-12-02 12:24:41,457 | INFO | Ruta C:\Users\gianlu\Market-Scraper\Market-Scraper encontrada.
2025-12-02 12:24:41,457 | INFO | CSV exportado hacia C:\Users\gianlu\Market-Scraper\Market-Scraper\data\processed\job_market_processed.csv
2025-12-02 12:24:42,292 | INFO | CSV creado en data\processed\job_market_processed.csv
